In [34]:
import glob
import os
import urllib.parse
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from fink_utils.photometry.conversion import dc_mag

# Rubin KN candidate filters

Used data transfer system
- Alerts between August 1st 2024 and 2025: **43,221,791 alerts, 2267.07GB**

## Alert class (manually introduced)
SIMBAD("galaxy"),
SIMBAD("Galaxy"),
SIMBAD("EmG"),
SIMBAD("Seyfert"),
alert_classes = [
    "(SIMBAD) Galaxy",
    "(SIMBAD) EmO",
    "(SIMBAD) Seyfert",
    "(SIMBAD) Seyfert1",
    "(SIMBAD) Seyfert2",
    "(SIMBAD) Seyfert_1",
    "(SIMBAD) Seyfert_2",
    "(SIMBAD) BlueCompG",
    "(SIMBAD) BlueCompactG",
    "(SIMBAD) StarburstG",
    "(SIMBAD) LSB_G",
    "(SIMBAD) HII_G",
    "(SIMBAD) GinCl",
    "(SIMBAD) GinGroup",
    "(SIMBAD) GinPair",
    "(SIMBAD) BCIG",
    "(SIMBAD) PartofG",
    "(SIMBAD) Compact_Gr_G",
    "(SIMBAD) IG",
    "(SIMBAD) PairG",
    "(SIMBAD) GroupG",
    "(SIMBAD) CIG",
    "(SIMBAD) SuperClG",
    "(SIMBAD) Void",
    "(SIMBAD) LINER",
    "(SIMBAD) Possible_CIG",
    "(SIMBAD) Possible_G",
    "(SIMBAD) Possible_GrG",
    "(SIMBAD) Possible_SCIG",
    "(SIMBAD) GravLens",
    "(SIMBAD) GravLensSystem",
    "Unknown",
    "(Fink) Kilonova candidates",
    "(Fink) Supernova candidates",
    "(Fink) Early Supernova Ia candidates",
    "(Fink) Microlensing candidates",
    "(Fink) Tracklet (space debris & satellite glints)",
    "(Fink) Ambiguous"
]

## Extra conditions
-- 1. Remove potential subtraction artefacts
-- 2. Far from known solar system objects
-- 3. Eliminate roids
-- 4. Recent detections (less than 20 days old)

candidate.drb > 0.9;
candidate.classtar > 0.4;
(candidate.ssdistnr > 10 OR candidate.ssdistnr < 0);
roid != 3;
(candidate.jd - candidate.jdstarthist) < 20;

**alerts after these cuts 456876 alerts**

In [2]:
# NUmber of original alerts for the year
n_alerts_year = 43221791

# Load data

In [3]:


# Load all parquet files from data/Fink/ subdirectories
parquet_files = glob.glob('data/Fink/**/*.parquet', recursive=True)
print(f"Found {len(parquet_files)} parquet files")

if len(parquet_files) == 0:
    print("No parquet files found in data/Fink/")
else:
    dfs = []
    for i, f in enumerate(parquet_files):
        try:
            df_temp = pd.read_parquet(f)
            
            # Extract finkclass from folder name
            folder_path = os.path.dirname(f)
            folder_name = os.path.basename(folder_path)
            
            if folder_name.startswith('finkclass='):
                # Extract after 'finkclass=' and decode URL encoding
                finkclass = urllib.parse.unquote(folder_name.split('finkclass=')[1])
            else:
                finkclass = folder_name
            
            # Skip if it's directly in data/Fink/ (no meaningful subfolder)
            if finkclass == 'Fink':
                continue
                
            # Add finkclass column
            df_temp['finkclass'] = finkclass
            dfs.append(df_temp)
            
            if (i + 1) % 500 == 0:
                print(f"Processed {i+1}/{len(parquet_files)} files")
                
        except Exception as e:
            print(f"Error reading {f}: {e}")
    
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        print(f"Loaded {len(df)} alerts from {len(dfs)} files")
        print(f"Shape: {df.shape}")
        print(f"Finkclass distribution:")
        print(df['finkclass'].value_counts())
        print(f"Sample of first few columns: {df.columns.tolist()[:10]}")
    else:
        print("No files could be read successfully")

Found 2869 parquet files
Processed 500/2869 files
Processed 1000/2869 files
Processed 1500/2869 files
Processed 2000/2869 files
Processed 2500/2869 files
Loaded 456876 alerts from 2869 files
Shape: (456876, 50)
Finkclass distribution:
finkclass
Unknown                   362958
SN candidate               48675
Tracklet                   42218
Early SN Ia candidate       1702
Ambiguous                    705
Kilonova candidate           251
Seyfert1                     117
GinCl                         74
Seyfert2                      49
Microlensing candidate        36
LINER                         33
ClG                           14
Seyfert_1                     13
EmG                           10
Seyfert_2                      6
LSB_G                          6
Seyfert                        4
GinGroup                       2
PairG                          1
BClG                           1
PartofG                        1
Name: count, dtype: int64
Sample of first few columns: ['magps

In [4]:
df.groupby('tnsclass').size().sort_values(ascending=False)

tnsclass
Unknown                    443328
(TNS) SN Ia                  8404
(TNS) SN II                  2341
(TNS) CV                      392
(TNS) SN Ic                   316
(TNS) SN IIn                  288
(TNS) SN Ia-91T-like          276
(TNS) SN Ic-BL                207
(TNS) SN Ib                   174
(TNS) SN IIb                  163
(TNS) SLSN-I                  125
(TNS) SN IIP                  109
(TNS) TDE                     102
(TNS) SN Iax[02cx-like]        91
(TNS) SN Ia-pec                86
(TNS) Nova                     75
(TNS) SN Ia-91bg-like          69
(TNS) SLSN-II                  63
(TNS) SN                       53
(TNS) SN Ibn                   49
(TNS) SN Ib/c                  37
(TNS) TDE-He                   35
(TNS) Galaxy                   17
(TNS) NA/Unknown               17
(TNS) SN Ia-CSM                16
(TNS) TDE-featureless          14
(TNS) SN Ia-SC                 10
(TNS) TDE-H-He                  8
(TNS) SN I                      5
(TNS)

In [5]:
df.keys().to_list()

['magpsf',
 'sigmapsf',
 'fid',
 'jd',
 'ra',
 'dec',
 'ssnamenr',
 'candid',
 'schemavsn',
 'publisher',
 'objectId',
 'brokerIngestTimestamp',
 'brokerStartProcessTimestamp',
 'fink_broker_version',
 'fink_science_version',
 'cdsxmatch',
 'DR3Name',
 'Plx',
 'e_Plx',
 'vsx',
 'spicy_id',
 'spicy_class',
 'gcvs',
 'x3hsp',
 'x4lac',
 'mangrove',
 'roid',
 'rf_snia_vs_nonia',
 'snn_snia_vs_nonia',
 'snn_sn_vs_all',
 'mulens',
 'nalerthist',
 'rf_kn_vs_nonkn',
 't2',
 'anomaly_score',
 'lc_features_g',
 'lc_features_r',
 'jd_first_real_det',
 'jdstarthist_dt',
 'mag_rate',
 'sigma_rate',
 'lower_rate',
 'upper_rate',
 'delta_time',
 'from_upper',
 'brokerEndProcessTimestamp',
 'tracklet',
 'timestamp',
 'tnsclass',
 'finkclass']

### Basic cuts

In [18]:
# BASIC CUTS - Applied in data transfer + galactic plane filter locally

# 1. Remove all alerts which are potential subtraction artefacts
# mask_high_drb = df['drb'].astype(float) > 0.9
# mask_high_classtar = df['classtar'].astype(float) > 0.4
# mask_artefacts = mask_high_drb & mask_high_classtar

# 2. Far from known solar system objects
# mask_far_from_mpc = (df['ssdistnr'].astype(float) > 10) | (df['ssdistnr'].astype(float) < 0)

# 3. Recent detections (less than 20 days old)
# mask_new_detection = df['jd'].astype(float) - df['jdstarthist'].astype(float) < 20

# 4. Eliminate roids (if roid column exists)
if 'roid' in df.columns:
    mask_no_roids = df['roid'] != 3
else:
    mask_no_roids = pd.Series([True] * len(df), index=df.index)
    print("Warning: 'roid' column not found, skipping roid filter")

# 5. Away from galactic plane (|b| > 10 degrees)
coords = SkyCoord(df['ra'].astype(float), df['dec'].astype(float), unit="deg")
b = coords.galactic.b.deg
mask_away_from_galactic_plane = np.abs(b) > 20

# 6. Keep only extragalactic host candidates
# keep_cds = return_list_of_eg_host()
# if 'cdsxmatch' in df.columns:
#     mask_known_objects = df['cdsxmatch'].isin(keep_cds)
# else:
#     mask_known_objects = pd.Series([True] * len(df), index=df.index)
#     print("Warning: 'cdsxmatch' column not found, skipping SIMBAD filter")

# Combined basic cuts
# mask_basic_cuts = (mask_artefacts & mask_far_from_mpc & mask_new_detection & 
#                    mask_no_roids & mask_away_from_galactic_plane & mask_known_objects)
mask_basic_cuts = (mask_no_roids & mask_away_from_galactic_plane)

# Create basic cuts dataframe
df_basic_cuts = df[mask_basic_cuts].copy()

print(f"Basic cuts summary:")
print(f"  Original alerts: {len(df)}")
# print(f"  After drb > 0.9: {mask_high_drb.sum()}")
# print(f"  After classtar > 0.4: {mask_high_classtar.sum()}")
# print(f"  After artefact removal: {mask_artefacts.sum()}")
# print(f"  After MPC distance: {(mask_artefacts & mask_far_from_mpc).sum()}")
# print(f"  After recent detection: {(mask_artefacts & mask_far_from_mpc & mask_new_detection).sum()}")
# print(f"  After roid filter: {(mask_artefacts & mask_far_from_mpc & mask_new_detection & mask_no_roids).sum()}")
# print(f"  After galactic plane: {(mask_artefacts & mask_far_from_mpc & mask_new_detection & mask_no_roids & mask_away_from_galactic_plane).sum()}")
print(f"  Final basic cuts: {len(df_basic_cuts)}")
print(f"  Basic candidates per month: {len(df_basic_cuts)/12:.0f}")
print(f"  Basic cuts vs yearly alert number: {len(df_basic_cuts)/n_alerts_year*100:.2f}%")


Basic cuts summary:
  Original alerts: 456876
  Final basic cuts: 286915
  Basic candidates per month: 23910
  Basic cuts vs yearly alert number: 0.66%


THIS IS TOO MUCH FOR LVK REPROCESSING

What are these alerts?

In [19]:
def show_finkclass_distribution(df, title="Finkclass distribution"):
    """Display count and percentage distribution of finkclass column"""
    finkclass_counts = df['finkclass'].value_counts()
    finkclass_percent = df['finkclass'].value_counts(normalize=True) * 100
    finkclass_summary = pd.DataFrame({
        'Count': finkclass_counts,
        'Percentage': finkclass_percent.round(2)
    })
    print(f"{title}:")
    print(finkclass_summary)
    return finkclass_summary

# Show distribution after basic cuts
finkclass_summary_basic = show_finkclass_distribution(df_basic_cuts, "Finkclass distribution after basic cuts")

Finkclass distribution after basic cuts:
                         Count  Percentage
finkclass                                 
Unknown                 215573       75.13
SN candidate             35909       12.52
Tracklet                 33184       11.57
Early SN Ia candidate     1242        0.43
Ambiguous                  552        0.19
Seyfert1                   117        0.04
Kilonova candidate         101        0.04
GinCl                       74        0.03
Seyfert2                    46        0.02
LINER                       33        0.01
Microlensing candidate      27        0.01
Seyfert_1                   13        0.00
ClG                         13        0.00
EmG                         10        0.00
Seyfert_2                    6        0.00
LSB_G                        6        0.00
Seyfert                      4        0.00
GinGroup                     2        0.00
PairG                        1        0.00
BClG                         1        0.00
PartofG      

In [32]:
def plot_lightcurves(idxs, df):
    """Plot lightcurves for given objectIds from df"""
    for idx in idxs:
        plt.figure(figsize=(3,2))
        for i, row in df[df['objectId']==idx].iterrows():
            plt.errorbar(row['jd'], row['magpsf'],  fmt='o', label=row['fid'])
        plt.xlabel("JD")
        plt.ylabel("Magnitude")
        plt.title(f"{idx}")
        plt.legend()
    plt.show()
def plot_hist(df,xvar='magpsf',log=False):
    """Plot histograms for given DataFrame"""
    plt.figure(figsize=(5,2))
    plt.hist(df[xvar], bins=30, alpha=0.7)
    plt.xlabel(xvar)
    plt.ylabel("Count")
    if log:
        plt.yscale('log')
    plt.legend()
    plt.show()
import plotly.graph_objects as go

def plot_candidates_mollweide(df):
    """Plot candidates on Mollweide projection"""
    if len(df) == 0:
        print("No candidates to plot")
        return
    
    # Get coordinates
    ra = df['ra'].values
    dec = df['dec'].values
    
    # Wrap RA for Mollweide projection
    ra_wrapped = (ra + 180) % 360 - 180
    
    # Create color mapping based on finkclass
    finkclass_colors = {
        'Unknown': 'blue',
        'SN candidate': 'red', 
        'Seyfert1': 'green',
        'Seyfert_1': 'orange',
        'Kilonova candidate': 'purple'
    }
    
    colors = [finkclass_colors.get(fc, 'gray') for fc in df['finkclass']]
    
    fig = go.Figure()
    
    # Add scatter plot of Silver candidates
    fig.add_trace(go.Scattergeo(
        lon=ra_wrapped,
        lat=dec,
        mode='markers',
        marker=dict(
            size=8,
            color=colors,
            line=dict(width=1, color='black')
        ),
        hoverinfo='text',
        text=[f'ObjectId: {oid}<br>RA: {r:.2f}°<br>Dec: {d:.2f}°<br>Class: {fc}<br>JD: {jd:.1f}' 
              for oid, r, d, fc, jd in zip(df['objectId'], ra, dec, 
                                         df['finkclass'], df['jd'])],
        name='Silver Candidates'
    ))
    
    # Add galactic plane (±10 degrees)
    l_vals = np.linspace(0, 360, 1000)
    b_upper = 10 * np.ones_like(l_vals)
    b_lower = -10 * np.ones_like(l_vals)

    
    upper_edge = SkyCoord(l=l_vals * u.deg, b=b_upper * u.deg, frame='galactic').icrs
    lower_edge = SkyCoord(l=l_vals[::-1] * u.deg, b=b_lower[::-1] * u.deg, frame='galactic').icrs
    
    galactic_ra = np.concatenate([upper_edge.ra.deg, lower_edge.ra.deg])
    galactic_dec = np.concatenate([upper_edge.dec.deg, lower_edge.dec.deg])
    galactic_ra_wrapped = (galactic_ra + 180) % 360 - 180
    
    fig.add_trace(go.Scattergeo(
        lon=galactic_ra_wrapped,
        lat=galactic_dec,
        mode='lines',
        fill='toself',
        fillcolor='rgba(200, 100, 100, 0.3)',
        line=dict(width=0),
        hoverinfo='skip',
        showlegend=True,
        name='Galactic Plane (±10°)'
    ))
    
    fig.update_geos(
        projection_type="mollweide",
        showland=False,
        showcoastlines=False,
        lataxis_showgrid=True,
        lonaxis_showgrid=True,
    )
    
    fig.update_layout(
        title=f'Candidates Sky Distribution ({len(df)} candidates)',
        geo=dict(resolution=50, showframe=False),
        height=400,
        width=800,
        margin={"r":0,"t":50,"l":0,"b":0}
    )
    
    fig.show()

In [30]:
# idxs = df[df['finkclass']=='SN candidate'].objectId.unique()
# print(idxs)
# plot_lightcurves(idxs[:5], df)

In [22]:
df['lum_dist_values'] = pd.to_numeric(
    df['mangrove'].apply(lambda x: x['lum_dist']), 
    errors='coerce'
)

# All sky

In [ ]:
# BRONZE TIER: Basic cuts + very recent detections (< 3 days)
mask_very_recent = df['jdstarthist_dt'].astype(float) < 3
mask_nalert = df['nalerthist']<6 # Keep low alert history
mask_deltat = df['delta_time'] < 3 # Keep very recent detections
mask_tracklet = df['finkclass'] != "Tracklet" # Eliminate potential roids/satellites
mask_plx = df['Plx']/df['e_Plx']<5 #eliminate Gaia well measured parallaxes
mask_gcvs = df['gcvs'].isin(["Unknown","nan"])
mask_vsx = df['vsx'].isin(["Unknown","nan",'Fail 503','Fail 504','Fail 500','Fail 502'])
mask_bronze = mask_basic_cuts & mask_very_recent & mask_nalert & mask_tracklet & mask_plx & mask_gcvs & mask_vsx

df_bronze = df[mask_bronze].copy()

print(f"Bronze tier:")
print(f"  Basic cuts: {len(df_basic_cuts)}")
print(f"  Bronze candidates: {len(df_bronze)}")
print(f"  Bronze candidates per month: {len(df_bronze)/12:.0f}")
print(f"  Bronze efficiency: {len(df_bronze)/len(df_basic_cuts)*100:.2f}% of basic cuts")
print(f"  Bronze cuts vs yearly alert number: {len(df_bronze)/n_alerts_year*100:.2f}%")


Bronze tier:
  Basic cuts: 286915
  Bronze candidates: 51237
  Bronze candidates per month: 4270
  Bronze efficiency: 17.86% of basic cuts
  Bronze cuts vs yearly alert number: 0.12%


In [24]:
finkclass_summary_bronze = show_finkclass_distribution(df_bronze, "Finkclass distribution after bronze cuts")  

Finkclass distribution after bronze cuts:
                    Count  Percentage
finkclass                            
Unknown             49878       97.35
SN candidate         1289        2.52
Seyfert1               44        0.09
Seyfert2                8        0.02
LINER                   6        0.01
Seyfert_1               6        0.01
Seyfert                 3        0.01
Kilonova candidate      2        0.00
EmG                     1        0.00


In [36]:
df_bronze.magpsf.describe()

count    51237.000000
mean        18.883481
std          0.853931
min         11.940845
25%         18.376759
50%         18.909136
75%         19.403185
max         21.835711
Name: magpsf, dtype: float64

In [33]:
plot_candidates_mollweide(df_bronze)

# Galaxy targeted

In [25]:
# Silver TIER: Basic cuts + very recent detections (< 2 days)
mask_very_recent = df['jdstarthist_dt'].astype(float) < 2
mask_mangrove = df['lum_dist_values'] > 0
mask_silver = mask_bronze & mask_very_recent & mask_mangrove

df_silver = df[mask_silver].copy()

print(f"Silver tier:")
print(f"  Basic cuts: {len(df_basic_cuts)}")
print(f"  Silver candidates: {len(df_silver)}")
print(f"  Silver candidates per month: {len(df_silver)/12:.0f}")
print(f"  Silver efficiency: {len(df_silver)/len(df_basic_cuts)*100:.2f}% of basic cuts")
print(f"  Silver cuts vs yearly alert number: {len(df_silver)/n_alerts_year*100:.2f}%")


Silver tier:
  Basic cuts: 286915
  Silver candidates: 357
  Silver candidates per month: 30
  Silver efficiency: 0.12% of basic cuts
  Silver cuts vs yearly alert number: 0.00%


In [37]:
df_silver.magpsf.describe()

count    357.000000
mean      18.908490
std        0.870541
min       16.237268
25%       18.395025
50%       18.890871
75%       19.298782
max       21.557470
Name: magpsf, dtype: float64

In [26]:
finkclass_summary_silver = show_finkclass_distribution(df_silver, "Finkclass distribution after silver cuts")  


Finkclass distribution after silver cuts:
              Count  Percentage
finkclass                      
Unknown         336       94.12
SN candidate     11        3.08
Seyfert1          6        1.68
Seyfert_1         4        1.12


In [31]:


# Plot Silver candidates on sky
plot_candidates_mollweide(df_silver)

# Gold

In [ ]:


# GOLD TIER: Bronze + rate calculation from Fink KN filter
rate_results = []
df_sel = df_silver.copy()

# Group by objectId and process each object
for obj_id in df_sel['objectId'].unique():
    obj_alerts = df_sel[df_sel['objectId'] == obj_id].sort_values('jd')
    
    if len(obj_alerts) < 1:
        continue
    
    # Get photometric history arrays for this object
    magpsf_hist = obj_alerts['magpsf'].iloc[0]  # Array of magnitudes
    sigmapsf_hist = obj_alerts['sigmapsf'].iloc[0]  # Array of errors
    jd_hist = obj_alerts['jd'].values  # Julian dates
    
    # Handle case where magpsf_hist might not be an array
    if not isinstance(magpsf_hist, (list, np.ndarray)):
        continue
        
    # Remove NaN values
    mask_valid = ~(pd.isna(magpsf_hist) | pd.isna(sigmapsf_hist))
    if mask_valid.sum() < 2:
        continue
    
    mag_hist = np.array(magpsf_hist)[mask_valid]
    err_hist = np.array(sigmapsf_hist)[mask_valid]
    jd_hist_clean = jd_hist[mask_valid]
    
    # Remove abnormal values (mag > 21)
    mask_good = mag_hist < 21
    if mask_good.sum() < 2:
        continue
        
    mag_hist = mag_hist[mask_good]
    err_hist = err_hist[mask_good]  
    jd_hist_clean = jd_hist_clean[mask_good]
    
    # Calculate rate using Fink method (last two measurements)
    if len(mag_hist) >= 2 and (jd_hist_clean[-1] - jd_hist_clean[-2]) > 0:
        dmag = mag_hist[-1] - mag_hist[-2]
        dt = jd_hist_clean[-1] - jd_hist_clean[-2]
        rate = dmag / dt
        error_rate = np.sqrt(err_hist[-1]**2 + err_hist[-2]**2) / dt
        
        rate_results.append({
            'objectId': obj_id, 
            'rate': rate, 
            'error_rate': error_rate,
            'dt': dt
        })

# Create Gold mask based on rate > 0.3 mag/day
rate_df = pd.DataFrame(rate_results)
rising_objects = rate_df[rate_df['rate'] < 0.3]['objectId'] if len(rate_df) > 0 else []
mask_gold = df_sel['objectId'].isin(rising_objects)

df_gold = df_sel[mask_gold].copy()

print(f"Gold tier summary:")
print(f"  Selection candidates: {len(df_sel)}")
print(f"  Objects with rate calculation: {len(rate_results)}")
print(f"  Objects with rate <0.3 mag/day: {len(rising_objects)}")
print(f"  Gold candidates: {len(df_gold)}")

if len(df_gold) > 0:
    show_finkclass_distribution(df_gold, "Gold tier finkclass distribution")

Gold tier summary:
  Selection candidates: 415
  Objects with rate calculation: 0
  Objects with rate <0.3 mag/day: 0
  Gold candidates: 0
